# 03 — Run the no-memory baseline

This is the first live model run. It sends each of the 12 fixed prompts to Gemini through its REST API. Persistent memory, retrieval, external web access, and tools are disabled. The notebook stores actual responses and measured latency/token data. This is a model-level baseline; a later notebook will run the tool-enabled Dexter agent separately.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
RESULTS_PATH = RUNS_DIR / 'results.jsonl'
RUNNER_PATH = EXPERIMENTS_DIR / 'run_no_memory_baseline.py'
if not MANIFEST_PATH.is_file():
    raise RuntimeError('Run 01_build_task_manifest.ipynb first.')
if not os.environ.get('GOOGLE_API_KEY'):
    raise RuntimeError('Set GOOGLE_API_KEY in this Jupyter process before running this notebook.')
print('Manifest:', MANIFEST_PATH)
print('Model: gemini-3.8-flash')
print('This baseline makes 12 paid Gemini API calls and writes real results.')

In [ ]:
# Run only once per condition. The runner skips existing run IDs, so it is safe to restart after interruption.
environment = os.environ.copy()
environment['BASELINE_MODEL'] = 'gemini-3.8-flash'
run = subprocess.run(
    [sys.executable, str(RUNNER_PATH), str(MANIFEST_PATH), str(RESULTS_PATH)],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=False,
)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('Baseline runner stopped. Existing completed task records were preserved.')

In [ ]:
results = [json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line]
baseline = [record for record in results if record['condition'] == 'none']
print(f'Completed baseline records: {len(baseline)} / 12')
for record in baseline:
    print(record['task_id'], '|', record['latency_ms'], 'ms |', record['total_tokens'], 'tokens |', record['answer_path'])
if len(baseline) != 12:
    raise RuntimeError('Baseline incomplete. Re-run the previous cell; completed tasks will be skipped.')